# STEP 2 - Competitive Landscape

| Question to answer (Presentation deck, Step 2) | How we measure it in this notebook |
|---|---|
| How concentrated is the field, and is it consolidating? | Six measures of concentration from the literature (Sections 3–4); the Herfindahl index and the 4-firm ratio over time (Section 5.1) |
| Who are the incumbents, and who are the new entrants? | Top applicants with type and country; new applicants per year, their share of families, and the rate of disappearance (Section 5.2) |
| Which subfields are growing fastest? | Growth of the CPC groups of the sector between two periods (Section 5.3) |
| Which political, economic, social and technological forces shape it? | PEST analysis of the political, economic, social and technological forces, with sources outside PATSTAT (Section 5.4) |

**Contents**

1. Setup and configuration
2. The unit of analysis: applicants and harmonisation, i.e. Entity Linking
3. Proxies for concentration: six measures from the literature
4. Example: inventive concentration in fuel cells (Liston-Heyes & Pilkington, 2004)
5. Answering the Step 2 questions for our sector

## 1. Setup and configuration

In [1]:
%pip install -q cleanco

Note: you may need to restart the kernel to use updated packages.


In [23]:
import re
import sys
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy import stats
from sqlalchemy import func, or_, and_
from cleanco import basename

# ---------------------------------------------------------------- environment & config
REPO = Path.cwd() if (Path.cwd() / "config").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

from stage00_config import load_config
cfg = load_config()

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN, TLS202_APPLN_TITLE, TLS203_APPLN_ABSTR,
    TLS206_PERSON, TLS207_PERS_APPLN, TLS209_APPLN_IPC, 
    TLS211_PAT_PUBLN, TLS224_APPLN_CPC
)

ENV = cfg.env
OUTPUT_DIR =  Path("outputs/step2")  
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------- applicant names (Section 2)
NAME_METHOD = "cleanco"           # "psn" (PATSTAT names as they are), "dictionary" or "cleanco" or "terrorizer"
GOLD_STANDARD_FILE = "PATSTAT_500_label_datav4.csv"   # hand-labelled names (Section 2.5), in the same folder as this notebook

# ---------------------------------------------------------------- replication (Section 4): the paper's design
#REPL_OFFICE = "US"                                      # USPTO patents...
#REPL_GRANTED = (date(1976, 1, 1), date(2002, 12, 31))   # ...granted between these two dates
#REPL_KEYWORDS = ("fuel cell",)                          # searched in titles and abstracts
#REPL_CLASSES = ("H01M8",)                               # IPC/CPC H01M 8: fuel cells
#REPL_SELECTION = "keywords or classes"                  # "keywords", "classes" or "keywords or classes"
#PAPER_COUNTRIES = ("AU", "CA", "FR", "DE", "IL", "IT", "JP", "NL", "SE", "CH", "GB", "US")


# ---------------------------------------------------------------- sector parameters
IPR_TYPE = "PI"                      # patents of invention only
SECTOR_NAME = cfg.sector_name
SPLIT_YEAR = 2015                    # incumbents file before SPLIT_YEAR, entrants from SPLIT_YEAR on
WINDOW = 3                           # years per window when following concentration over time
TOP_N = 30                           # rows in the tables of top applicants, i.e number of players selected

COUNTRY_NAMES = {"AU": "Australia", "CA": "Canada", "FR": "France", "DE": "Germany", "IL": "Israel", "IT": "Italy",
                 "JP": "Japan", "NL": "Netherlands", "SE": "Sweden", "CH": "Switzerland", "GB": "UK", "US": "USA", "CN": "China", "KR": "South Korea"}

patstat = PatstatClient(env=ENV)
db = patstat.orm()
print(f"Sector: {SECTOR_NAME} | env: {ENV}")

Sector: Quantum computing | env: PROD


## 2. The unit of analysis: applicants

In Step 1 we counted **families** and **applications**. Concentration needs one more unit: the **applicant**, the owner of the patent. We must decide when two names on two patents are **the same applicant**.

In [24]:
def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


def keyword_condition(terms, exclusions=()):
    """Any term appears in the title or abstract and no exclusion phrase does (case-insensitive).

    A missing title or abstract is read as empty text: NOT applied to a NULL comparison is NULL,
    which would drop the application silently.
    """
    title = func.lower(func.coalesce(TLS202_APPLN_TITLE.appln_title, ""))
    abstract = func.lower(func.coalesce(TLS203_APPLN_ABSTR.appln_abstract, ""))
    match = or_(*(col.like(f"%{t.lower()}%") for t in terms for col in (title, abstract)))
    if not exclusions:
        return match
    excluded = or_(*(col.like(f"%{x.lower()}%") for x in exclusions for col in (title, abstract)))
    return and_(match, ~excluded)

                          
family_id = TLS201_APPLN.docdb_family_id.label("docdb_family_id")
by_cpc = (db.query(family_id)
          .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
          .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, cfg.cpc_prefixes)))

by_kw = (db.query(family_id)
         .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
         .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
         .filter(keyword_condition(cfg.keywords)))

sector_ids = by_cpc.union(by_kw)

# Prepare the subquery for families within year_min and year_max
family_year = func.min(TLS201_APPLN.earliest_filing_year)
families = (
    db.query(TLS201_APPLN.docdb_family_id.label("unit_id"), family_year.label("year"))
    .filter(TLS201_APPLN.ipr_type == cfg.ipr_type,
            TLS201_APPLN.docdb_family_id.in_(sector_ids.subquery().select()))
    .group_by(TLS201_APPLN.docdb_family_id)
    .having(family_year.between(cfg.year_min, cfg.year_max))
    .subquery()
)

# Extract all the applicant names for these families
q_sector = (
    db.query(families.c.unit_id, families.c.year, TLS206_PERSON.psn_id, TLS206_PERSON.psn_name,
             TLS206_PERSON.psn_sector, func.trim(TLS206_PERSON.person_ctry_code).label("ctry"))
    .select_from(families)
    .join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == families.c.unit_id)
    .join(TLS207_PERS_APPLN, TLS207_PERS_APPLN.appln_id == TLS201_APPLN.appln_id)
    .join(TLS206_PERSON, TLS206_PERSON.person_id == TLS207_PERS_APPLN.person_id)
    .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)
    .distinct()
)
raw_sector = patstat.df(q_sector)
print(f"Extracted {raw_sector['unit_id'].nunique():,} patent families with their respective applicants.")

Extracted 12,280 patent families with their respective applicants.


### 2.1 The "name game"

The same company appears in PATSTAT under many spellings (*SIEMENS AG*, *Siemens Aktiengesellschaft*, *SIEMENS AKTIENGESELLSCHAFT MUENCHEN*…). If we count each spelling as a different firm, a large firm is split into many small ones and **every concentration measure is wrong**. `tls206_person` offers:

| Column | What it is |
|---|---|
| `person_name` | the name exactly as printed on the document: never use it to count applicants |
| `psn_id`, `psn_name` | the **PATSTAT Standardised Name** (EPO and K.U. Leuven): a first harmonisation, available for every person |
| `han_id`, `han_name` | the **OECD HAN** (Harmonised Applicant Names): a stronger harmonisation of firm names |

`psn_name` is a good start, but it still leaves variants (*TOYOTA JIDOSHA KK* and *TOYOTA MOTOR CORP*, with and without legal forms…), and it never merges a subsidiary with its parent.

### 2.2 Three possible ways to consolidate names further

| Method | How it works | Good at | Misses |
|---|---|---|---|
| **A · Dictionary of legal forms** | upper-case the name, remove punctuation and the legal-form words (INC, LTD, GMBH, AG, KK…) | variants that differ only in punctuation or legal form | translations, abbreviations, groups |
| **B · `cleanco`** | a Python package with the legal forms ("business entity" extensions) of many countries | the same, also abbreviations with dots (S.p.A., N.V.) | the same |
| **C · Terrorizer** | NLP and network techniques: name similarity, a knowledge base of company names, community detection on the network of similar names (https://github.com/graziasveva/Terrorizer) | spelling variants, abbreviations, names in different forms | it needs a check of the groups it creates |

Whatever the method, **look at the top of your list by hand**: in the end, you decide what one applicant is.

**Method A** is a few lines of code: a set of legal-form words, and a function that removes them from the beginning and the end of a name.

In [25]:
# Legal-form words to remove (extend the set for the countries of your sector)
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED", "LLC", "LP", "PLC",
    "GMBH", "AG", "KG", "MBH", "SE", "SA", "SAS", "SPA", "SRL", "BV", "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA", "PTY", "THE", "AKTIENGESELLSCHAFT", "GESELLSCHAFT",
    "GCV", "CV", "SARL", "SRO", "PTE", "BHD", "OOO", "ZAO", "OAO", "PJSC", "JSC", "LLP", "ULC", "UG", "SL"}


def clean_with_dictionary(name):
    "Upper-case, keep letters and digits only, and drop legal-form words at the start and at the end."
    words = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:      # e.g. "TOYOTA JIDOSHA KABUSHIKI KAISHA" -> "TOYOTA JIDOSHA"
        words.pop()
    while words and words[0] in LEGAL_FORMS:       # e.g. "KABUSHIKI KAISHA TOSHIBA" -> "TOSHIBA"
        words.pop(0)
    return " ".join(words)


examples = ["SIEMENS AG", "Siemens Aktiengesellschaft", "BALLARD POWER SYSTEMS INC.", "Ballard Power Systems",
            "TOYOTA JIDOSHA KABUSHIKI KAISHA", "KABUSHIKI KAISHA TOSHIBA", "Toyota Motor Corp",
            "FIAT S.p.A.", "Umicore N.V."]
pd.DataFrame({"name": examples, "method A": [clean_with_dictionary(n) for n in examples]})

,name,method A
0,SIEMENS AG,SIEMENS
1,Siemens Aktiengesellschaft,SIEMENS
2,BALLARD POWER SYSTEMS INC.,BALLARD POWER SYSTEMS
3,Ballard Power Systems,BALLARD POWER SYSTEMS
4,TOYOTA JIDOSHA KABUSHIKI KAISHA,TOYOTA JIDOSHA
5,KABUSHIKI KAISHA TOSHIBA,TOSHIBA
6,Toyota Motor Corp,TOYOTA MOTOR
7,FIAT S.p.A.,FIAT S P A
8,Umicore N.V.,UMICORE N V


**Method B** uses the `cleanco` package, which knows the legal forms of many countries, also when they are written as abbreviations with dots: method A turns *FIAT S.p.A.* into *FIAT S P A*, cleanco into *FIAT*. We use cleanco first, then the cleaning of method A.

In [26]:
def clean_with_cleanco(name):
    "Remove legal forms with cleanco, then apply the same upper-case cleaning as method A."
    return clean_with_dictionary(basename(str(name)))


pd.DataFrame({"name": examples,
              "method A": [clean_with_dictionary(n) for n in examples],
              "method B": [clean_with_cleanco(n) for n in examples]})

,name,method A,method B
0,SIEMENS AG,SIEMENS,SIEMENS
1,Siemens Aktiengesellschaft,SIEMENS,SIEMENS
2,BALLARD POWER SYSTEMS INC.,BALLARD POWER SYSTEMS,BALLARD POWER SYSTEMS
3,Ballard Power Systems,BALLARD POWER SYSTEMS,BALLARD POWER SYSTEMS
4,TOYOTA JIDOSHA KABUSHIKI KAISHA,TOYOTA JIDOSHA,TOYOTA JIDOSHA
5,KABUSHIKI KAISHA TOSHIBA,TOSHIBA,TOSHIBA
6,Toyota Motor Corp,TOYOTA MOTOR,TOYOTA MOTOR
7,FIAT S.p.A.,FIAT S P A,FIAT
8,Umicore N.V.,UMICORE N V,UMICORE


**Method C** uses NLP and network techniques (such as name similarity and community detection on networks of similar names, via Terrorizer https://github.com/graziasveva/Terrorizer) to handle spelling variants, abbreviations, and names written in entirely different forms. While powerful for merging complex variants, it always requires a manual check of the generated groups.

In [27]:
%pip install --upgrade terrorizer
from terrorizer import terrorizer_disambiguator

ERROR: Could not find a version that satisfies the requirement terrorizer (from versions: none)
ERROR: No matching distribution found for terrorizer
Note: you may need to restart the kernel to use updated packages.


ModuleNotFoundError: No module named 'terrorizer'

In [ ]:
df = pd.DataFrame({
    "name": examples,
    "method A": [clean_with_dictionary(n) for n in examples],
    "method B": [clean_with_cleanco(n) for n in examples]
})

try:
    results = terrorizer_disambiguator(df, column_name='method B')
    # If the function returns a dataframe, we extract the disambiguated column
    df["method C"] = results.iloc[:, -1] 
except Exception as e:
    # Fallback placeholder in case the package fails to load or requires a knowledge base file
    df["method C"] = df["method B"]
    print(f"Terrorizer execution note: {e}")

df

### 2.3 From persons to applicants (`TLS206_PERSON`): our harmonisation


- **`applicant`**: the consolidated name, with the method chosen in `NAME_METHOD`;
- **`type`**: firm, university, individual… from `psn_sector` (the column mixes several labels, so we map it to a few types)
- **`ctry`**: the applicant's country. A firm can have different countries on different patents (or none): we give each applicant the country that appears **most often** among its patents.

In [28]:
import unicodedata

# ---------------------------------------------------------------- specific cleaning
ABBREVIATIONS = {   # same organisation, different spelling or language
    "UNIV": "UNIVERSITY", "UNIVERSITE": "UNIVERSITY", "UNIVERSITAT": "UNIVERSITY", "UNIVERSITAET": "UNIVERSITY",
    "UNIVERSITA": "UNIVERSITY", "UNIVERSIDAD": "UNIVERSITY", "UNIVERSIDADE": "UNIVERSITY", "UNIVERSITEIT": "UNIVERSITY",
    "INST": "INSTITUTE", "INSTITUT": "INSTITUTE", "INSTITUTO": "INSTITUTE", "ISTITUTO": "INSTITUTE",
    "NATL": "NATIONAL", "NAT": "NATIONAL", "LAB": "LABORATORY", "LABS": "LABORATORIES", "TECH": "TECHNOLOGY",
    "TECHNOLOGIES": "TECHNOLOGY", "INTL": "INTERNATIONAL", "CENT": "CENTER", "CENTRE": "CENTER", "RES": "RESEARCH",
    "SYST": "SYSTEMS", "SCI": "SCIENCE", "SCIENCES": "SCIENCE", "ELECTR": "ELECTRONICS",
}
FILLER_WORDS = {"OF", "THE", "AND", "DE", "DEL", "DELLA", "DI", "DU", "DES", "LA", "LE", "DER", "DIE", "DAS",
                "FUR", "FUER", "FOR", "AT", "IN"}
TRAILING_NOISE = {"HOLDING", "HOLDINGS", "GROUP", "IP"}
NAME_PREFIXES = [   # institutional wrappers that hide the real owner
    r"^(THE )?BOARD OF TRUSTEES OF (THE )?", r"^(THE )?BOARD OF REGENTS( OF)?( THE)? ", r"^(THE )?REGENTS OF (THE )?",
    r"^(THE )?TRUSTEES OF (THE )?", r"^PRESIDENT AND FELLOWS OF ", r"^THE GOVERNING COUNCIL OF (THE )?",
    r"^UNITED STATES OF AMERICA AS REPRESENTED BY (THE )?", r"^(THE )?GOVERNMENT OF ",
]
# Manual aliases for the big players of the sector: (regex on the cleaned name, canonical name)
ORG_ALIASES = [
    (r"^INTERNATIONAL BUSINESS MACHINES|^IBM\b",       "IBM"),
    (r"^GOOGLE\b",                                      "GOOGLE"),
    (r"^MICROSOFT\b",                                   "MICROSOFT"),
    (r"^D WAVE\b|^DWAVE\b",                             "D-WAVE SYSTEMS"),
    (r"^HEWLETT PACKARD",                               "HEWLETT PACKARD"),
    (r"^NIPPON TELEGRAPH|^NTT\b",                       "NTT"),
    (r"^ORIGIN QUANTUM",                                "ORIGIN QUANTUM"),
    (r"^CENTER NATIONAL RECHERCHE|^CENTRE NATIONAL RECHERCHE|CNRS", "CNRS"),
    (r"^COMMISSARIAT ENERGIE ATOMIQUE",                 "CEA"),
    (r"^ELECTRONIC SCIENCE RESEARCH INSTITUTE",         "ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA"),
    (r"^INFORMATION SCIENCE ACADEMY",                   "INFORMATION SCIENCE ACADEMY CHINA"),
    (r"^BEIJING BAIDU NETCOM|^BAIDU\b",                    "BAIDU"),
    (r"^SGCC\b|^STATE GRID CORPORATION|^STATE GRID CORP",  "STATE GRID CORPORATION OF CHINA"),
    (r"^MIT\b|^MASSACHUSETTS INSTITUTE TECHNOLOGY",        "MIT"),
    (r"^ETRI\b|^ELECTRONICS TELECOMMUNICATIONS RESEARCH INSTITUTE", "ETRI"),
    (r"^HUAWEI",                                           "HUAWEI"),
    (r"^QUANTUMCTEK|^QUANTUM CTEK",                        "QUANTUMCTEK"),
    (r"^RIGETTI",                                          "RIGETTI"),
    (r"^QUANTINUUM",                                       "QUANTINUUM"),
    (r"^IONQ",                                             "IONQ"),
    (r"^INTEL\b",                                          "INTEL"),
    (r"^TOSHIBA\b",                                        "TOSHIBA"),
]

_ORG = [(re.compile(p), c) for p, c in ORG_ALIASES]


def ascii_upper(name):
    "Upper-case and remove accents (Müller -> MULLER, Université -> UNIVERSITE)."
    s = str(name).replace("ß", "ss").replace("Ø", "O").replace("ø", "o").replace("Æ", "AE").replace("æ", "ae")
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return s.upper().replace("&", " AND ")


def specific_cleaning(name):
    "Brackets, wrappers, abbreviations, filler words, trailing noise, word order of universities."
    s = ascii_upper(name)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    s = " ".join(s.split())
    for p in NAME_PREFIXES:
        s = re.sub(p, "", s)
    words = [ABBREVIATIONS.get(w, w) for w in s.split()]
    words = [w for w in words if w not in FILLER_WORDS]
    while words and words[-1] in TRAILING_NOISE:
        words.pop()
    if "UNIVERSITY" in words:                                  # OXFORD UNIVERSITY == UNIVERSITY OF OXFORD
        words = ["UNIVERSITY"] + [w for w in words if w != "UNIVERSITY"]
    return " ".join(words)


def my_harmonisation(name, use_aliases=True):
    "A (dictionary) + B (cleanco) + specific cleaning + manual aliases of the sector's big players."
    if name is None or str(name).strip() in ("", "nan", "None"):
        return ""
    base = clean_with_cleanco(ascii_upper(name))   # apply method B and then method A
    base = specific_cleaning(base)                 # add specific cleaning
    base = clean_with_dictionary(base)             # apply again method A
    if use_aliases:
        for rx, canon in _ORG:
            if rx.search(base):
                return canon
    return base

In [29]:
ORG_WORDS = {"TECHNOLOGY", "SYSTEMS", "QUANTUM", "COMPUTING", "LABORATORY", "LABORATORIES", "SOLUTIONS",
             "SEMICONDUCTOR", "ELECTRONICS", "SOFTWARE", "DEVICES", "NETWORKS", "INDUSTRIES", "ENERGY",
             "RESEARCH", "SCIENCE", "COMMUNICATIONS", "MICRO", "DIGITAL", "DATA"}
ACADEMIC_WORDS = {"UNIVERSITY", "COLLEGE", "ACADEMY", "FACULTY", "SCHOOL", "POLYTECHNIC"}
GOV_WORDS = {"INSTITUTE", "NATIONAL", "CENTER", "MINISTRY", "COMMISSARIAT", "FOUNDATION", "COUNCIL",
             "GOVERNMENT", "LABORATORY", "DEPARTMENT"}


def applicant_type(psn_sector, raw_name=""):
    "Type from psn_sector; when it is empty, infer it from the name."
    s = str(psn_sector).upper()
    if "UNIVERSITY" in s or any(t in s for t in ["ACADEMY", "COLLEGE", "FACULTY"]): return "university"
    if "HOSPITAL" in s: return "hospital"
    if any(t in s for t in ["GOV", "MINISTRY", "LABORATORY", "PUBLIC", "RESEARCH INSTITUTE"]): return "government / non-profit"
    if "INDIVIDUAL" in s: return "individual"
    if "COMPANY" in s or "FIRM" in s: return "firm"
    # psn_sector missing: guess from the name
    words = set(re.sub(r"[^A-Z0-9 ]", " ", ascii_upper(raw_name)).split())
    if words & ACADEMIC_WORDS: return "university"
    if words & GOV_WORDS: return "government / non-profit"
    if words & (LEGAL_FORMS - {"THE", "CO"}) or words & ORG_WORDS: return "firm"
    if 2 <= len(words) <= 4: return "individual"      # "HOU XIAOYAN", "ROSE GEORDIE"
    return "unknown"

def consolidated_names(psn_names, use_aliases=True):
    "Dictionary {PATSTAT name: consolidated name}."
    return {n: my_harmonisation(n, use_aliases) for n in sorted(set(psn_names))}


def prepare_applicants(raw, use_aliases=True):
    "From (unit_id, year, person) rows to one row per (unit_id, applicant) with type and country."
    df = raw.copy()
    df = df.dropna(subset=["psn_name"])
    df = df[df["psn_name"].str.strip() != ""]

    # 1. consolidated name
    df["applicant"] = df["psn_name"].map(consolidated_names(df["psn_name"], use_aliases))
    df = df[df["applicant"].str.strip() != ""]

    # 2. type: from psn_sector or from the name, then the most frequent type of each applicant
    df["type"] = [applicant_type(s, n) for s, n in zip(df["psn_sector"], df["psn_name"])]
    known_t = df[df["type"] != "unknown"]
    if not known_t.empty:
        main_type = known_t.groupby("applicant")["type"].agg(lambda t: t.value_counts().index[0])
        df["type"] = df["applicant"].map(main_type).fillna("unknown")

    # 3. one country per applicant: the most frequent known one
    df["ctry"] = df["ctry"].replace("", np.nan)
    known = df.dropna(subset=["ctry"]).drop_duplicates(["unit_id", "applicant", "ctry"])
    if not known.empty:
        main_c = known.groupby("applicant")["ctry"].agg(lambda c: c.value_counts().index[0])
        df["ctry"] = df["applicant"].map(main_c)

    # 4. one row per (patent, applicant)
    return df[["unit_id", "year", "applicant", "type", "ctry"]].drop_duplicates(["unit_id", "applicant"])

Now, we can apply the harmonisation directly on our applicants: we adapt method A and B to our own sector, adding some specific harmonisation.

In [30]:
df_applicants = prepare_applicants(raw_sector)

print(f"Total rows after cleaning and deduplication: {len(df_applicants)}")
print(f"Unique harmonised applicants: {df_applicants['applicant'].nunique()}")
print(f"Raw PATSTAT names: {raw_sector['psn_name'].nunique():,}")
print(df_applicants["type"].value_counts())
print("-" * 50)

df_applicants.head(15)

Total rows after cleaning and deduplication: 17287
Unique harmonised applicants: 4488
Raw PATSTAT names: 5,384
type
firm                       9387
university                 3495
individual                 3304
government / non-profit    1012
unknown                      89
Name: count, dtype: int64
--------------------------------------------------


,unit_id,year,applicant,type,ctry
1,68480559,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
2,68480760,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
3,68480568,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
4,75188684,2020,INFORMATION SCIENCE ACADEMY CHINA,university,NaN
5,68480589,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
6,68480607,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
7,68480742,2019,ELECTRONIC SCIENCE RESEARCH INSTITUTE CHINA,firm,NaN
8,34954098,2004,CNRS,government / non-profit,FR
9,50883643,2012,KOREA RESEARCH INSTITUTE STANDARD SCIENCE,government / non-profit,KR
10,54063171,2015,UNIVERSITY OXFORD INNOVATION,university,GB


### 2.4 Top-N audit for Quantum Computing by number of families

In [31]:
# Audit: top applicants by number of families, with the PATSTAT names merged into each one
top = df_applicants.groupby("applicant")["unit_id"].nunique().sort_values(ascending=False).head(TOP_N)
variants = (raw_sector.assign(applicant=raw_sector["psn_name"].map(consolidated_names(raw_sector["psn_name"].dropna())))
            .groupby("applicant")["psn_name"].agg(lambda x: " | ".join(sorted(set(x.dropna())))[:200]))
audit = top.rename("families").to_frame().join(variants.rename("merged names"))
#audit.to_csv(OUTPUT_DIR / "audit_top_applicants.csv")
audit

,families,merged names
applicant,,
IBM,598,IBM (CHINA) INVESTMENT COMPANY | IBM (INTERNAT...
ORIGIN QUANTUM,560,"ORIGIN QUANTUM COMPUTING COMPANY, LIMITED, HEF..."
MICROSOFT,214,MICROSOFT CORPORATION | MICROSOFT LICENSING TE...
D-WAVE SYSTEMS,198,D-WAVE (COMMERCIAL) | D-WAVE SYSTEMS | D-WAVE ...
QUANTUMCTEK,198,QUANTUMCTEK (BEIJING) COMPANY | QUANTUMCTEK (G...
TOSHIBA,180,TOSHIBA CORPORATION | TOSHIBA DIGITAL SOLUTION...
BAIDU,179,BEIJING BAIDU NETCOM SCIENCE AND TECHNOLOGY CO...
GOOGLE,176,GOOGLE | GOOGLE LIMITED RESPONSIBILITY COMPANY...
INTEL,153,INTEL CORPORATION


### 2.5 Test our method: harmonise the names of a gold standard

How good is our harmonisation method? To know, we need names for which the right answer is already known: a **gold standard**.

**The data.** The file `PATSTAT_500_label_datav4.csv` (in the same folder as this notebook) contains applicant names as they appear in PATSTAT, grouped by hand. It has two columns:

| Column | What it is |
|---|---|
| `comp` | an applicant name, as written in PATSTAT |
| `Group` | the identifier of the **real organisation** behind the name: names with the same `Group` belong to the same organisation |

#### How to measure how well you did: precision, recall and F1

We compare your groups with the gold groups by looking at **pairs of names**. For every pair of names:

| | same gold group | different gold groups |
|---|---|---|
| **same predicted group** (you merged them) | true positive (TP) | false positive (FP) |
| **different predicted groups** (you kept them apart) | false negative (FN) | true negative |

$$\text{Precision} = \frac{TP}{TP + FP} \qquad \text{Recall} = \frac{TP}{TP + FN} \qquad F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

- **Precision**: of the pairs you merged, the share that really belong together. It is low when you **merge too much** (two different firms become one).
- **Recall**: of the pairs that belong together, the share you merged. It is low when you **merge too little** (one firm stays split in many names).
- **F1**: the harmonic mean of the two; it is high only when both are high.

**Example.** The gold standard has one organisation with the names {*NOKIA*, *NOKIA CORP*, *NOKIA OYJ*} (3 pairs) and *NOKIAN TYRES* alone. Your method puts all four names together: you merged 6 pairs, 3 of them right. Precision = 3/6 = 0.5, recall = 3/3 = 1, F1 = 0.67.

**Other metrics you can use** to compare the groups:

- **number of groups found**, against the true number of organisations;
- **homogeneity** (each predicted group contains one organisation only), **completeness** (each organisation ends up in one predicted group) and their harmonic mean, the **V-measure** (`sklearn.metrics.homogeneity_completeness_v_measure`);

Each metric looks at the errors from a different angle.

In [37]:
# load the hand-labelled file (columns: comp = name, Group = real organisation)
GOLD_PATH = REPO / "notebooks" / GOLD_STANDARD_FILE
assert GOLD_PATH is not None, f"{GOLD_STANDARD_FILE} not found: put it next to the notebook or in data/"

gold = pd.read_csv(GOLD_PATH, sep=None, engine="python", encoding_errors="replace")   # sep=None: detects , or ;
missing = {"comp", "Group"} - set(gold.columns)
assert not missing, f"Missing columns {missing}: the file has {list(gold.columns)}"

gold = (gold.rename(columns={"comp": "name", "Group": "gold_group"})[["name", "gold_group"]]
            .dropna().astype({"name": str}).reset_index(drop=True))
gold["name"] = gold["name"].str.strip()

print(f"Gold standard: {len(gold)} names | {gold['gold_group'].nunique()} real organisations")
print(f"Names repeated with different groups: "
      f"{(gold.groupby('name')['gold_group'].nunique() > 1).sum()}")
gold.head(10)

Gold standard: 3192 names | 481 real organisations
Names repeated with different groups: 0


,name,gold_group
0,NOKIA CORPORATION,1
1,NOKIA (DEUTSCHLAND),1
2,NOKIA GRAETZ,1
3,NOKIA SIEMENS NETWORKS SCHWEIZ,2
4,NOKIA,1
5,NOKIA MOBILE PHONES (U.K.) LIMITED,1
6,NOKIA SIEMENS NETWORKS S.P.A.,2
7,NOKIA SIEMENS NETWORKS,2
8,NOKIA COMMUNICATION,1
9,NOKIA HIGH SPEED ACCESS PRODUCTS,1


In [38]:
from math import comb


def pairs_inside(group_sizes):
    "Number of pairs of names inside groups of the given sizes: n(n-1)/2 for each group."
    return sum(comb(int(n), 2) for n in group_sizes)


def precision_recall_f1(gold_groups, predicted_groups):
    "Pairwise precision, recall and F1 of a predicted grouping against the gold groups."
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())         # merged by you and in the gold standard
    tp_fp = pairs_inside(df.groupby("pred").size())                # all the pairs you merged
    tp_fn = pairs_inside(df.groupby("gold").size())                # all the pairs that belong together
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}

In [39]:
from sklearn.metrics import homogeneity_completeness_v_measure

def evaluate(f):
    "Pairwise precision/recall/F1 + V-measure of a name -> group function on the gold standard."
    pred = gold["name"].map(f)
    res = precision_recall_f1(gold["gold_group"], pred)
    h, c, v = homogeneity_completeness_v_measure(gold["gold_group"], pred)
    res.update({"homogeneity": h, "completeness": c, "V-measure": v})
    return res, pred

methods = {
    "psn (no change)":                    lambda n: str(n).strip(),
    "A  dictionary":                      clean_with_dictionary,
    "B  cleanco + dictionary":            clean_with_cleanco,
    "A+B + specific cleaning":            lambda n: my_harmonisation(n, use_aliases=False),
    "A+B + specific + aliases (diagn.)":  lambda n: my_harmonisation(n, use_aliases=True),
}
results, preds = {}, {}
for label, f in methods.items():
    results[label], preds[label] = evaluate(f)

comparison = pd.DataFrame(results).T
comparison["true groups"] = gold["gold_group"].nunique()
comparison.round(3)

/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)
/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% 

,groups found,precision,recall,F1,homogeneity,completeness,V-measure,true groups
psn (no change),3192.0,1.000,0.000,0.000,1.000,0.593,0.745,481
A dictionary,3007.0,0.995,0.002,0.005,1.000,0.600,0.750,481
B cleanco + dictionary,2983.0,0.996,0.003,0.005,1.000,0.601,0.750,481
A+B + specific cleaning,2937.0,0.994,0.004,0.007,1.000,0.602,0.752,481
A+B + specific + aliases (diagn.),2797.0,0.981,0.055,0.104,0.999,0.616,0.762,481


In [40]:
best = "A+B + specific cleaning"
df_err = gold.assign(pred=preds[best].values)

# False positives: predicted groups that contain more than one real organisation (merged too much)
fp = (df_err.groupby("pred").filter(lambda g: g["gold_group"].nunique() > 1)
            .sort_values(["pred", "gold_group"])[["pred", "gold_group", "name"]])
# False negatives: real organisations split into several predicted groups (merged too little)
fn = (df_err.groupby("gold_group").filter(lambda g: g["pred"].nunique() > 1)
            .sort_values(["gold_group", "pred"])[["gold_group", "pred", "name"]])

print(f"{best}: {fp['pred'].nunique()} predicted groups mix different organisations (precision errors), "
      f"{fn['gold_group'].nunique()} organisations are split (recall errors)")
print("\n--- Merged too much (false positives) ---")
display(fp.head(40))
print("\n--- Merged too little (false negatives) ---")
display(fn.head(40))

A+B + specific cleaning: 2 predicted groups mix different organisations (precision errors), 179 organisations are split (recall errors)

--- Merged too much (false positives) ---


,pred,gold_group,name
2855,SCHAEFFLER,469,SCHAEFFLER
2866,SCHAEFFLER,470,SCHAEFFLER GROUP
2618,SUMITOMO CHEMICAL,373,SUMITOMO CHEMICAL
2610,SUMITOMO CHEMICAL,375,SUMITOMO CHEMICAL COMPANY



--- Merged too little (false negatives) ---


,gold_group,pred,name
0,1,NOKIA,NOKIA CORPORATION
4,1,NOKIA,NOKIA
17,1,NOKIA,NOKIA LIMITED
37,1,NOKIA,NOKIA UK LIMITED
20,1,NOKIA ALUMIINI,NOKIA ALUMIINI
58,1,NOKIA AMERICA,NOKIA OF AMERICA CORPORATION
8,1,NOKIA COMMUNICATION,NOKIA COMMUNICATION
43,1,NOKIA COMMUNICATIONS,NOKIA COMMUNICATIONS
22,1,NOKIA CONSUMER ELECTRONICS,NOKIA CONSUMER ELECTRONICS
42,1,NOKIA COPPORATION,NOKIA COPPORATION
